# 技能（Skill）抽象与管理：技能池（Skills Pool）

---

## 课程导航

| 部分 | 内容 | 对应代码 |
|------|------|----------|
| **Part 1** | 为什么需要「技能层」 | — |
| **Part 2** | 环境准备与项目结构 | `backend/skills/` |
| **Part 3** | 技能抽象：BaseSkill + 统一输入输出 | `backend/skills/base_skill.py` |
| **Part 4** | 注册表：把技能「管起来」 | `backend/skills/registry.py` |
| **Part 5** | 内置技能：真实执行的 4 个能力 | `backend/skills/mock_skills.py` |
| **Part 6** | 自定义技能：skill.md 驱动的「配置化技能」 | `backend/skills/skills_manager.py` |
| **Part 7** | 生命周期管理：CRUD / zip 导入 / 开关持久化 | `backend/skills/skills_manager.py` |
| **Part 8** | 接入 API 与前端 | `backend/api/routes.py` + `frontend/` |
| **Part 9** | 课程总结 + 课后练习 | — |

---

> **面向学员**：本教程面向弱基础学员，每个知识点都有详细讲解。
> 建议按顺序逐 Cell 运行。这不是「背代码」，而是理解「代码为什么这么写」。
> 前置依赖：第 22 节（工具调用闭环已就绪）。

Jupyter 异步运行说明

本 notebook 在 Jupyter 内核的自动事件循环中运行，异步调用直接写顶层 `await`，
不要用 `asyncio.run()`（会报 `RuntimeError`）。

- ✅ 正确：`await demo.run_skill(...)`
- ❌ 错误：`asyncio.run(...)`

> ⚠️ 本节的**写操作演示**（新增 / 编辑 / 删除 / 导入）都在**临时目录**里进行，
> 演示结束会自动还原，不会污染你真实的 `backend/skills_pool/` 与 `skill_config.json`。

---
## Part 1：为什么需要「技能层」
---

### 1.1 回顾：第 22 节我们有了什么

第 22 节实现了完整的工具调用闭环：

| 层 | 内容 |
|----|------|
| LLM 封装 | `llm/client.py`：配置化、懒加载单例 |
| 内置工具 | `builtin_tools.py`：current_time / calculator（注册表 + Schema + 开关） |
| MCP | `web_search_server.py` + `mcp_client.py`：联网搜索等远程能力 |
| 循环 | `single_agent.py`：Function Calling 主循环（≤5 轮） |

但这里管的是「工具（Tool）」——**单个函数**。

### 1.2 工具与技能的区别

> 生活类比：**工具是螺丝刀，技能是「会修水管的水电工」**。
> 水电工要会选工具、看现场、按步骤施工 —— 这是一组动作的组合，不是单一函数。

| | 工具 Tool | 技能 Skill |
|---|----------|-----------|
| 粒度 | 单一函数（current_time） | 完整能力单元（搜索 → 分析 → 出结论） |
| 载体 | Python 函数 + JSON Schema | skill.md 说明 +（可选）skill.py 实现 |
| 心智 | 「怎么执行」 | 「是什么能力、什么场景用、怎么用」 |
| 示例 | `calculator` | 「去 AI 味」的 humanizer 技能 |

### 1.3 本节课的演进路线

| 版本 | 做什么 | 对应代码 |
|------|--------|---------|
| v0 | 没有抽象，技能是散落的特例 | 手写函数 |
| v1 | 统一基类 BaseSkill + 输入输出契约 | `skills/base_skill.py` |
| v2 | 注册表统一管理（内置禁删） | `skills/registry.py` |
| v3 | 内置技能真实执行（搜索 / 分析 / 写作 / 审核） | `skills/mock_skills.py` |
| v4 | 自定义技能 = 一份 skill.md（配置化技能） | `skills/skills_manager.py` |
| v5 | 生命周期管理：CRUD / zip 导入 / 开关持久化 | `skills/skills_manager.py` |
| v6 | Claude 技能包兼容：references 注入 + scripts 工具循环 | `skills/skills_manager.py` |

### 1.4 本节交付成果

- ✅ `BaseSkill` / `SkillInput` / `SkillOutput` 技能抽象
- ✅ `SkillRegistry` 注册表（内置禁删）
- ✅ 4 个内置技能（真实执行）
- ✅ `skill.md` 自定义技能 + `.zip` 导入
- ✅ 技能开关（统一持久化 `skill_config.json`，重启保留）
- ✅ `/api/v1/skills` CRUD 接口 + 前端「技能中心」

In [1]:
# ============================================================
# Cell 1：环境准备 —— 把 backend 加入模块搜索路径
# ============================================================
import os, sys, inspect, json
from pathlib import Path

print('=' * 55)
print('  环境检查')
print('=' * 55)
print(f'  Python 版本: {sys.version.split()[0]}')
print(f'  工作目录: {os.getcwd()}')

# 自动定位 backend/（不依赖当前工作目录）
# 教学要点：sys.path 是 Python 查找模块的搜索路径列表。找不到模块报
# ModuleNotFoundError，多半是这一步没执行，或工作目录不在 smartbrief/ 下。
# 这里先"向上"找含 backend/ 的父目录，找不到再"向下"找常见子目录。
def _find_root(start: Path) -> Path:
    p = start
    while True:
        if (p / 'backend').exists():
            return p
        if p.parent == p:
            break
        p = p.parent
    for sub in sorted(start.iterdir()):
        if sub.is_dir() and (sub / 'backend').exists():
            return sub
    raise RuntimeError('未找到 backend/ 目录，请把工作目录切换到 smartbrief/')

root = _find_root(Path.cwd())
sys.path.insert(0, str(root / 'backend'))
sys.path.insert(0, str(root))

# 验证第 23 节相关模块可导入
from skills.base_skill import BaseSkill, SkillInput, SkillOutput
from skills import registry as _registry
from skills import mock_skills as mock_skills_mod
from skills import skills_manager as sm
print('  [OK] 技能抽象层导入成功')

print(f'  技能池目录 : {sm.SKILLS_POOL_DIR}')
print(f'  开关配置文件: {sm.SKILL_CONFIG_PATH}')
print()
print('  提示：请从 Cell 1 开始按顺序运行本 notebook')

  环境检查
  Python 版本: 3.10.20
  工作目录: c:\Users\13691\Documents\Code\260601-GPKJ\05_multi-agent\proj02\smartbrief
  [OK] 技能抽象层导入成功
  技能池目录 : C:\Users\13691\Documents\Code\260601-GPKJ\05_multi-agent\proj02\smartbrief\backend\skills_pool
  开关配置文件: C:\Users\13691\Documents\Code\260601-GPKJ\05_multi-agent\proj02\smartbrief\backend\skill_config.json

  提示：请从 Cell 1 开始按顺序运行本 notebook


---
## Part 2：项目结构（第 23 节相关部分）
---

第 23 节的代码集中在 `backend/skills/` 与 `backend/skills_pool/`：

| 路径 | 作用 |
|------|------|
| `backend/skills/base_skill.py` | 技能抽象：SkillInput / SkillOutput / BaseSkill |
| `backend/skills/registry.py` | 技能注册表 SkillRegistry（全局单例） |
| `backend/skills/mock_skills.py` | 4 个内置技能（启动时注册） |
| `backend/skills/skills_manager.py` | 技能池管理器：CRUD / zip 导入 / 开关持久化 |
| `backend/skills_pool/<name>/skill.md` | 自定义技能持久化目录（skill.md 必需） |
| `backend/skill_config.json` | 全部技能开关状态（重启保留） |
| `backend/api/routes.py` | `/api/v1/skills` 系列路由 |
| `frontend/src/views/SkillCenter.vue` | 技能中心页面（本期只作了解） |

> 🔄 **加载顺序**（后端启动时）：
> ① 注册 4 个内置技能 → ② 扫描 `skills_pool/` 加载自定义技能 → ③ 应用 `skill_config.json` 里的开关状态。

---
## Part 3：技能抽象 —— BaseSkill + 统一输入输出
---

### 3.1 为什么先定义「输入 / 输出契约」？

> 类比：快递行业。各家快递的面单格式不一样，但收件端只认行业标准
> （快递单号 + 三段式地址）——这个标准就是「契约」。

调用方（AI 对话 / 前端 / 其他技能）不该关心每个技能内部的差异，只需要知道三件事：

1. **入参**：用户请求 `query` + 可选上下文 `context`
2. **出参**：统一 `status / data / metadata`
3. **元信息**：`name / description / category / version / builtin / enabled`

### 3.2 先看没抽象时的痛点（Cell 2），再看正式版（Cell 3）

In [2]:
# ============================================================
# Cell 2：v0 —— 没有抽象，每个技能都是「特例」
# ============================================================

# 假设你有两个能力：查天气、算字数
async def weather_skill(query):
    return {"status": "success", "data": {"city": query, "temp": 26}, "meta": {}}

async def word_count_skill(query):
    return {"status": "success", "data": {"count": len(query)}, "meta": {}}

# 调用方要「知道每个技能长什么样」
result = await weather_skill("上海")
print("weather 调用:", result)

print()
print("[问题] 如果再加 10 个技能：")
print("  1. 每个技能的入参 / 出参格式各写各的，上层没法统一处理")
print("  2. 想给技能加「分类、版本、开关」？得在每个函数里重复实现")
print("  3. 想枚举「我现在有哪些技能」？没有统一清单")

weather 调用: {'status': 'success', 'data': {'city': '上海', 'temp': 26}, 'meta': {}}

[问题] 如果再加 10 个技能：
  1. 每个技能的入参 / 出参格式各写各的，上层没法统一处理
  2. 想给技能加「分类、版本、开关」？得在每个函数里重复实现
  3. 想枚举「我现在有哪些技能」？没有统一清单


In [3]:
# ============================================================
# Cell 3：看正式版 —— BaseSkill 抽象 + 统一输入输出
# ============================================================
print('=' * 55)
print('  base_skill.py —— 技能基类')
print('=' * 55)
print(inspect.getsource(BaseSkill))

print('=' * 55)
print('  SkillInput / SkillOutput —— 统一的入参 / 出参契约')
print('=' * 55)
print(inspect.getsource(SkillInput))
print(inspect.getsource(SkillOutput))

# 一个最小实现示例：只有 execute() 是必须自己写的
class EchoSkill(BaseSkill):
    name = 'echo'
    display_name = '复读'
    description = '把用户的话原样返回'
    category = 'writing'
    version = '1.0.0'

    async def execute(self, input: SkillInput) -> SkillOutput:
        return SkillOutput(status='success', data={'reply': input.query})

echo = EchoSkill()
print()
print('[演示] 一个最小技能 to_dict() —— 给 API / 前端展示的统一格式')
print(json.dumps(echo.to_dict(), ensure_ascii=False, indent=2))

  base_skill.py —— 技能基类
class BaseSkill(ABC):
    """技能基类：所有技能（内置 / 自定义）必须继承并实现 execute()。

    字段说明：
    - name：技能唯一标识（如 mock_search），注册表以此为 key
    - display_name：面向用户的中文展示名（如"搜索"）
    - description：给 LLM / 用户看的技能说明
    - category：技能分类（search | analysis | writing | review）
    - version：版本号
    - builtin：True 表示内置技能（不可删除 / 修改）
    - enabled：开关状态（false 时不可被调用 / 试运行）
    """

    name: str = ""
    display_name: str = ""
    description: str = ""
    category: str = ""       # search | analysis | writing | review
    version: str = "1.0.0"
    builtin: bool = False    # 内置技能不可删除 / 修改
    enabled: bool = True     # 开关：false 时不可试运行 / 被智能体调用

    @abstractmethod
    async def execute(self, input: SkillInput) -> SkillOutput:
        """执行技能，返回统一输出（所有技能实现的核心方法）。"""
        raise NotImplementedError

    def to_dict(self) -> dict:
        """序列化给 API / 前端展示（内置与自定义技能统一用这套字段）。"""
        return {
            "name": self.name,
            "display_name": self.display_name,
            "descrip

---
## Part 4：注册表 —— 把技能「管起来」
---

### 4.1 需求

技能会越来越多（内置 + 自定义），需要一个统一入口管理：

| 方法 | 功能 |
|------|------|
| `register` | 注册技能（**重名要拦截**） |
| `get / get_all` | 按名字查 / 枚举全部 |
| `unregister` | 注销（**内置技能禁止删除**） |
| `get_source` | 来源：builtin / custom |

### 4.2 内置技能为什么禁删？

> 内置技能是框架自带的「地基能力」（搜索 / 分析 / 写作 / 审核）。
> 允许删除的话，一个误操作就能把系统删残。
> 商业产品里这叫「系统能力 vs 用户数据」的边界：
> **系统预置内容只读，用户内容可自由管理。**

### 4.3 演进：先看手动 dict（Cell 4）的问题，再看注册表（Cell 5）

In [4]:
# ============================================================
# Cell 4：v1 —— 用 dict 手动管理技能
# ============================================================

# 两个现成的技能类
class SearchSkill(BaseSkill):
    name = 'search'
    display_name = '搜索'
    description = '联网搜索'
    category = 'search'

    async def execute(self, input):
        return SkillOutput(status='success', data={'results': ['示例结果']})

class WriteSkill(BaseSkill):
    name = 'write'
    display_name = '写作'
    description = '写草稿'
    category = 'writing'

    async def execute(self, input):
        return SkillOutput(status='success', data={'draft': '示例草稿'})

# 手动 dict 管理
skills = {}
skills[SearchSkill.name] = SearchSkill()
skills[WriteSkill.name] = WriteSkill()

print('当前技能清单:', list(skills.keys()))
print()
print('[问题] 手动 dict 的痛点：')
print('  1. 注册逻辑散落在各处，没有统一收口')
print('  2. 「内置禁删」这种规则没有载体')
print('  3. 重名注册无人拦截，会静默覆盖')

当前技能清单: ['search', 'write']

[问题] 手动 dict 的痛点：
  1. 注册逻辑散落在各处，没有统一收口
  2. 「内置禁删」这种规则没有载体
  3. 重名注册无人拦截，会静默覆盖


In [5]:
# ============================================================
# Cell 5：看正式版 —— SkillRegistry 注册表
# ============================================================
from skills.registry import SkillRegistry

print('=' * 55)
print('  registry.py —— 技能注册表')
print('=' * 55)
print(inspect.getsource(SkillRegistry))

# 用它管理上面的两个技能
reg = SkillRegistry()
reg.register(SearchSkill())
reg.register(WriteSkill())
print()
print('[演示] 注册两个技能，get_all() 枚举：')
for s in reg.get_all():
    print(f'  • {s.name} [{s.category}] builtin={s.builtin}')

# 重名注册 -> 抛错
try:
    reg.register(SearchSkill())
    print('[NO] 重名竟然通过了')
except ValueError as exc:
    print(f'[OK] 重名注册被拦截: {exc}')

# 内置技能禁删：注册一个 builtin=True 的技能试试
class VersionSkill(BaseSkill):
    name = 'version'
    display_name = '版本'
    description = '查看系统版本'
    category = 'analysis'
    builtin = True

    async def execute(self, input):
        return SkillOutput(status='success', data={'version': 'v23.0'})

reg.register(VersionSkill())
try:
    reg.unregister('version')
    print('[NO] 内置技能竟然被删了')
except ValueError as exc:
    print(f'[OK] 内置技能不可删除: {exc}')

# 自定义技能可以删除
reg.unregister('search')
print('[OK] 自定义技能 search 删除成功，剩余:', [s.name for s in reg.get_all()])

  registry.py —— 技能注册表
class SkillRegistry:
    """技能注册表：name → BaseSkill。

    教学方法：这是一个"薄封装"类——内部就是一个 dict，
    所有方法都是对 dict 的语义化操作（注册 / 注销 / 查询 / 开关）。
    """

    def __init__(self) -> None:
        self._skills: dict[str, BaseSkill] = {}

    def clear(self) -> None:
        """清空注册表（重新初始化时使用，见 SkillsManager.init）。"""
        self._skills.clear()

    def register(self, skill: BaseSkill) -> None:
        """注册技能（重名抛错）。

        重名拦截很重要：技能名是唯一 key，若两个技能同名，
        后注册的会悄悄覆盖先注册的，导致"改了 A 却影响到 B"的隐蔽 bug。
        """
        if not skill.name:
            raise ValueError("技能 name 不能为空")
        if skill.name in self._skills:
            raise ValueError(f"技能已存在：{skill.name}")
        self._skills[skill.name] = skill

    def unregister(self, name: str) -> None:
        """注销技能（内置技能抛错，禁止删除）。"""
        skill = self._skills.get(name)
        if skill is None:
            raise KeyError(f"技能不存在：{name}")
        if skill.builtin:
            raise ValueError(f"内置技能不可删除：{name}")
        de

---
## Part 5：内置技能 —— 真实执行的 4 个能力
---

启动时注册 4 个内置技能，全部「真实执行」：

| 技能 | 分类 | 实现方式 | 说明 |
|------|------|---------|------|
| `mock_search` | search | 调用 MCP `web_search` 工具 | 真实联网（失败降级） |
| `mock_analyze` | analysis | 调用 LLM | 结构化分析结论 |
| `mock_write` | writing | 调用 LLM | 生成结构化草稿 |
| `mock_review` | review | 调用 LLM | 评分 + 修改建议 |

两个设计要点：

1. **统一输出**：无论成功失败都返回 `SkillOutput`（status: success / failed）
2. **失败不抛异常**：MCP / LLM 异常被捕获，转成 `status="failed"` + `metadata.error`
   —— 上层（API / AI 对话）只需判断 status，不需要给每个技能写 try/except

In [6]:
# ============================================================
# Cell 6：看内置技能 —— 真实执行的 4 个 mock 技能
# ============================================================
print('=' * 55)
print('  MockSearchSkill —— 调用 MCP web_search（真实联网）')
print('=' * 55)
print(inspect.getsource(mock_skills_mod.MockSearchSkill))

print('=' * 55)
print('  MOCK_SKILLS —— 启动时逐个注册的 4 个内置技能')
print('=' * 55)
for cls in mock_skills_mod.MOCK_SKILLS:
    print(f'  • {cls.name:14s} [{cls.category:8s}] {cls.description[:30]}')

print()
print('[提示] MockAnalyzeSkill / MockWriteSkill / MockReviewSkill 走 LLM，')
print('      失败同样返回 status="failed"，模式完全一致')

  MockSearchSkill —— 调用 MCP web_search（真实联网）
class MockSearchSkill(BaseSkill):
    """真实搜索：调用 MCP web_search 工具（DuckDuckGo），失败降级为模拟结果。"""

    name = "mock_search"
    display_name = "搜索"
    description = "联网搜索互联网，返回标题、摘要与链接列表"
    category = "search"
    version = "1.0.0"
    builtin = True

    async def execute(self, input: SkillInput) -> SkillOutput:
        # 技能内部跨进程调用 MCP：mcp_manager 会路由到 web_search_server 子进程
        try:
            result = await mcp_manager.call_tool(
                WEB_SEARCH_TOOL, {"query": input.query}
            )
            results = result.get("results", []) if isinstance(result, dict) else []
            return SkillOutput(
                status="success",
                data={"query": input.query, "results": results},
                metadata={"mode": "web", "source": result.get("source", "mcp")},
            )
        except Exception as exc:
            # 网络 / MCP 异常 → 返回 failed，而不是抛出
            return SkillOutput(
                status="fai

In [7]:
# ============================================================
# Cell 7：演示 —— 初始化技能池 + 试运行内置搜索技能
# ============================================================
from mcp_client import mcp_client as mc

print('=' * 55)
print('  ① 初始化技能池（等价于后端启动时 lifespan 做的事）')
print('=' * 55)
sm.manager.init()
for s in sm.manager.list_skills():
    tag = '[内置]' if s['builtin'] else '[自定义]'
    print(f'  {tag} {s["name"]:14s} [{s["category"]:8s}] builtin={str(s["builtin"]):5s} enabled={s["enabled"]}')

print()
print('=' * 55)
print('  ② 试运行 mock_search（调用 MCP 联网搜索）')
print('=' * 55)
# 先确保 MCP 内置 Server 已连接（与后端启动行为一致；已连接则跳过）
try:
    await mc.mcp_manager.start()
except Exception as exc:
    print('  [MCP] 启动失败:', exc)

out = await sm.manager.run_skill('mock_search', '什么是MCP协议？')
print(f'  status : {out.status}')
print(f'  metadata: {out.metadata}')
if out.data and out.data.get('results'):
    for r in out.data['results'][:3]:
        title = r.get('title') or r.get('url') or ''
        print(f'    - {str(title)[:46]}')
else:
    print('  （网络不可用时 status=failed，但不会抛异常 —— 这就是统一输出契约的好处）')

  ① 初始化技能池（等价于后端启动时 lifespan 做的事）
  [内置] mock_search    [search  ] builtin=True  enabled=True
  [内置] mock_analyze   [analysis] builtin=True  enabled=True
  [内置] mock_write     [writing ] builtin=True  enabled=True
  [内置] mock_review    [review  ] builtin=True  enabled=True
  [自定义] humanizer      [analysis] builtin=False enabled=True

  ② 试运行 mock_search（调用 MCP 联网搜索）
  status : failed
  metadata: {'error': '未知 MCP 工具：web_search（连接诊断：builtin-web-search: fileno）', 'mode': 'web'}
  （网络不可用时 status=failed，但不会抛异常 —— 这就是统一输出契约的好处）


---
## Part 6：自定义技能 —— 把技能「配置化」
---

### 6.1 核心洞察

上一节的技能是**写死在代码里**的类。但真实世界的技能大多数是「一份说明文档」：

> 你会用 Photoshop 吗？会。你背下它的源码了吗？没有。
> 你只是知道「怎么用」—— 使用说明（skill.md）就是技能本身。

自定义技能 = 一份 `skill.md`：

```markdown
---
name: humanizer
display_name: humanizer
description: 去除文本中的 AI 写作痕迹，让文字更像人类写作
category: analysis
version: 4.1.0
---

## 技能说明
（Markdown 正文：使用时机、处理流程、输出格式……）
```

- **frontmatter**（`---` 之间的键值对）：元数据 —— name / display_name / description / category / version
- **正文**：执行说明 —— 运行时被拼进 LLM 的 system prompt
- 可选 `skill.py`：需要确定性逻辑时提供 BaseSkill 子类；**缺失时由 `MarkdownSkill` 按 skill.md 让 LLM 执行**

### 6.2 开关为什么不写进 skill.md？

> 项目约定：开关状态统一存 `skill_config.json`（与第 22 节 `tool_config.json` 模式一致）。
> 原因：skill.md 是「技能的说明书」，可以被 zip 打包分发、被多人共享；
> 而「当前开 / 关」是运行态偏好，属于使用者，不属于技能本身。

In [8]:
# ============================================================
# Cell 8：v2 —— 手写一个简易 skill.md 解析器
# ============================================================
# 目标：把「技能说明文档」变成「技能元数据」
skill_text = """---
name: translator
display_name: 翻译官
description: 中英文互译
category: analysis
version: 1.0.0
---

请把用户输入的中文翻译成英文，英文翻译成中文。
"""

def parse_simple_frontmatter(text):
    meta, body = {}, text.strip()
    if not text.lstrip().startswith('---'):
        return meta, body
    lines = text.splitlines()
    end = None
    for i in range(1, len(lines)):
        if lines[i].strip() == '---':
            end = i
            break
    if end is None:
        return meta, body
    for line in lines[1:end]:
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        if ':' in line:
            key, _, val = line.partition(':')
            meta[key.strip()] = val.strip()
    body = '\n'.join(lines[end + 1:]).strip()
    return meta, body

meta, body = parse_simple_frontmatter(skill_text)
print('解析出的元数据:', meta)
print('正文前 20 字  :', body[:20])

解析出的元数据: {'name': 'translator', 'display_name': '翻译官', 'description': '中英文互译', 'category': 'analysis', 'version': '1.0.0'}
正文前 20 字  : 请把用户输入的中文翻译成英文，英文翻译成


In [9]:
# ============================================================
# Cell 9：看正式版 —— skills_manager 的 skill.md 解析与生成
# ============================================================
print('=' * 55)
print('  _parse_skill_md —— 解析 skill.md（frontmatter + 正文）')
print('=' * 55)
print(inspect.getsource(sm._parse_skill_md))

print('=' * 55)
print('  _build_skill_md —— 生成 skill.md（注意：不写 enabled）')
print('=' * 55)
print(inspect.getsource(sm._build_skill_md))

print('=' * 55)
print('  MarkdownSkill —— 没有 skill.py 时由 LLM 按说明执行')
print('=' * 55)
print(inspect.getsource(sm.MarkdownSkill))

print()
print('[提示] ① 无 references/scripts 时：execute 把 skill_md 拼进 system prompt')
print('       ② references/*.md 在构造时读入内存，随 system prompt 一并注入')
print('       ③ scripts/* 注册为 run_script 工具，走 Function Calling 工具循环')
print('       这就是「文档即技能」的实现原理（Claude 技能包资源支持）')

  _parse_skill_md —— 解析 skill.md（frontmatter + 正文）
def _parse_skill_md(text: str) -> tuple[dict, str]:
    """解析 skill.md：返回 (frontmatter 元数据, Markdown 正文)。

    教学要点——为什么是"手写"而不是 PyYAML：
    1) frontmatter 实际只有 name/display_name/description/category/version 等
       简单字段，手写十几行就能覆盖，避免为一个小功能引入依赖；
    2) 教学上正好讲清楚"键值对解析"的原理（partition + 首尾定界符）；
    3) 兼容 Claude 技能包（SKILL.md）的 YAML frontmatter 子集：
       - 引号值剥离（_unquote）：description: "..." → description
       - 多行值支持：
         a) 块标量：description: |（或 >），后续缩进行整体作为值；
         b) 隐式续行：无冒号但带缩进的行，归属上一个键折叠拼接。
    """
    meta: dict = {}
    body = text.strip()
    if text.lstrip().startswith("---"):
        lines = text.splitlines()
        end = None
        for i in range(1, len(lines)):
            if lines[i].strip() == "---":
                end = i
                break
        if end is not None:
            last_key: str | None = None
            block_mode: str | None = None  # "|"（保留换行）/ ">"（折叠空格）
            block_lines: list[str] = 

In [10]:
# ============================================================
# Cell 10：演示 —— 一份 skill.md 如何变成可运行技能
# ============================================================
# 复用 Cell 8 里的 skill_text（翻译官），用项目正式解析器
meta, body = sm._parse_skill_md(skill_text)
print('解析结果:', meta)

skill = sm.MarkdownSkill(meta, body)
print()
print('[演示] MarkdownSkill.to_dict():')
print(json.dumps(skill.to_dict(), ensure_ascii=False, indent=2))

print()
print('[可选] 真实执行（需 .env 配置 API Key；失败也返回 failed，不抛异常）')
out = await skill.execute(SkillInput(query='把 "hello world" 翻译成中文'))
print('  status:', out.status)
if out.data:
    print('  回复 :', str(out.data.get('reply', ''))[:80])
print()
print('[提示] 至此「技能 = 文档」的链路已经打通：')
print('       skill.md 文本  ->  MarkdownSkill 实例  -> 可执行')
print('       若技能目录含 references/ 或 scripts/，MarkdownSkill 会自动加载（Claude 技能包扩展）')

解析结果: {'name': 'translator', 'display_name': '翻译官', 'description': '中英文互译', 'category': 'analysis', 'version': '1.0.0'}

[演示] MarkdownSkill.to_dict():
{
  "name": "translator",
  "display_name": "翻译官",
  "description": "中英文互译",
  "category": "analysis",
  "version": "1.0.0",
  "builtin": false,
  "enabled": true,
  "source": "custom",
  "skill_md": "请把用户输入的中文翻译成英文，英文翻译成中文。"
}

[可选] 真实执行（需 .env 配置 API Key；失败也返回 failed，不抛异常）
  status: success
  回复 : 你好，世界

[提示] 至此「技能 = 文档」的链路已经打通：
       skill.md 文本  ->  MarkdownSkill 实例  -> 可执行
       若技能目录含 references/ 或 scripts/，MarkdownSkill 会自动加载（Claude 技能包扩展）


In [11]:
# ============================================================
# 补充演示：references 注入 + scripts 工具循环（Claude 技能包扩展）
# ============================================================
# 在临时目录构造一个"带资源"的技能包，看 MarkdownSkill 如何加载它们。
# 关键：传入 skill_dir 后，MarkdownSkill 会顺带扫描 references/ 与 scripts/。
import tempfile

with tempfile.TemporaryDirectory() as tmp:
    sk = Path(tmp) / 'my_skill'
    (sk / 'references').mkdir(parents=True)
    (sk / 'scripts').mkdir(parents=True)

    # skill.md：正文里声明"加载 references/禁用词表.md"
    (sk / 'skill.md').write_text('''---
name: my_skill
display_name: 示例技能
description: 演示 references 与 scripts 的加载
category: analysis
version: 1.0.0
---

请先阅读 references/禁用词表.md，再处理用户输入。
''', encoding='utf-8')

    # references：参考文档 → 构造时读入内存，随 system prompt 注入
    (sk / 'references' / '禁用词表.md').write_text(
        '绝对禁止使用：首先、其次、综上所述。', encoding='utf-8')

    # scripts：可执行脚本 → 白名单扫描，注册为 run_script 工具
    (sk / 'scripts' / 'stats.py').write_text(
        'import sys\nprint("词数:", len(sys.argv[1:]), "| 参数:", sys.argv[1:])\n',
        encoding='utf-8')

    meta, body = sm._parse_skill_md((sk / 'skill.md').read_text(encoding='utf-8'))
    skill = sm.MarkdownSkill(meta, body, skill_dir=sk)   # skill_dir 触发资源加载

    print('[1] 加载到的资源：')
    print('    references :', [name for name, _ in skill.references])
    print('    scripts    :', skill.scripts)

    print()
    print('[2] references 注入后的 system prompt：')
    print('    ' + skill._build_system().replace('\n', '\n    '))

    print()
    print('[3] run_script 工具 schema（script 用 enum 白名单）：')
    schema = skill._script_schema()
    print('    enum =', schema['function']['parameters']['properties']['script']['enum'])

    print()
    print('[4] 实际执行白名单脚本（禁 shell + 超时强杀）：')
    out = await skill._run_script('stats.py', ['hello', 'world'])
    print('    stdout:', out)

print()
print('[提示] references 内容对模型"可见"，scripts 供模型按需调用：')
print('       这就是 Claude 技能包 references/ 与 scripts/ 的落地支持')


[1] 加载到的资源：
    references : ['禁用词表.md']
    scripts    : ['stats.py']

[2] references 注入后的 system prompt：
    你是技能「示例技能」，请严格按照以下说明执行：
    
    请先阅读 references/禁用词表.md，再处理用户输入。
    
    【技能参考文档（references/，必须按需查阅）】
    ### 参考资料：禁用词表.md
    
    绝对禁止使用：首先、其次、综上所述。

[3] run_script 工具 schema（script 用 enum 白名单）：
    enum = ['stats.py']

[4] 实际执行白名单脚本（禁 shell + 超时强杀）：
    stdout: 词数: 2 | 参数: ['hello', 'world']

[提示] references 内容对模型"可见"，scripts 供模型按需调用：
       这就是 Claude 技能包 references/ 与 scripts/ 的落地支持


---
## Part 7：技能池生命周期管理 —— SkillsManager
---

### 7.1 管理接口一览

| 方法 | 对应 API | 功能 |
|------|---------|------|
| `list_skills()` | GET `/api/v1/skills` | 技能列表 |
| `create_skill(...)` | POST `/api/v1/skills` | 新增（表单 + skill.md 正文） |
| `update_skill(...)` | PUT `/api/v1/skills/{name}` | 编辑（内置返回 400） |
| `delete_skill(name)` | DELETE `/api/v1/skills/{name}` | 删除（内置返回 400） |
| `set_enabled(name, bool)` | PUT `/api/v1/skills/{name}/enabled` | 开关（统一持久化） |
| `import_skill(data)` | POST `/api/v1/skills/import` | zip 导入 |
| `run_skill(name, query)` | POST `/api/v1/skills/{name}/run` | 试运行（禁用技能返回 400） |

### 7.2 必须记住的校验与安全规则

| 规则 | 说明 |
|------|------|
| 技能名 | `^[a-zA-Z0-9][a-zA-Z0-9_-]{0,63}$`（字母开头，不能中文 / 空格） |
| 分类 | 仅 `search / analysis / writing / review / other` |
| zip 大小 | 上限 2MB（包体 + 解压后**双重检查**） |
| zip 路径 | 拒绝绝对路径 / `..` 路径穿越 / 盘符 |
| 内置技能 | 不可删除 / 不可修改（delete / update 返回 400） |
| 禁用技能 | 不可试运行（run 返回 400） |

> 💡 设计动机：zip 导入等于让用户上传「可执行目录」，
> 路径穿越（`../../evil.py`）是最经典的攻击面，必须逐条目校验。

In [12]:
# ============================================================
# Cell 11：看正式版 —— SkillsManager 关键源码
# ============================================================
print('=' * 55)
print('  create_skill —— 校验 -> 写 skill.md -> 注册 -> 持久化开关')
print('=' * 55)
print(inspect.getsource(sm.SkillsManager.create_skill))

print('=' * 55)
print('  delete_skill —— 内置禁删，自定义连目录一起删')
print('=' * 55)
print(inspect.getsource(sm.SkillsManager.delete_skill))

print('=' * 55)
print('  set_enabled —— 开关并统一持久化到 skill_config.json')
print('=' * 55)
print(inspect.getsource(sm.SkillsManager.set_enabled))

  create_skill —— 校验 -> 写 skill.md -> 注册 -> 持久化开关
    def create_skill(
        self, name, display_name, description="", category="analysis", version="1.0.0", skill_md="", enabled=True
    ) -> dict:
        name = name.strip()
        if not NAME_RE.match(name):
            raise ValueError("技能名仅支持字母、数字、下划线、中划线（且不能以 -/_ 开头）")
        if category not in VALID_CATEGORIES:
            raise ValueError(f"分类必须是 {'、'.join(VALID_CATEGORIES)} 之一")
        if _registry.registry.get(name) is not None:
            raise ValueError(f"技能已存在：{name}")

        body = (skill_md or "").strip() or DEFAULT_SKILL_BODY
        skill_dir = self._pool_dir() / name
        skill_dir.mkdir(parents=True, exist_ok=True)
        (skill_dir / "skill.md").write_text(
            _build_skill_md(name, display_name or name, description or "", category, version or "1.0.0", body),
            encoding="utf-8",
        )
        skill = MarkdownSkill(
            {"name": name, "display_name": display_name or name, "d

In [13]:
# ============================================================
# Cell 12：隔离演示 —— 在临时目录跑完整 CRUD（不污染真实技能池）
# ============================================================
import tempfile
from skills.skills_manager import SkillsManager

# 备份真实开关配置，演示结束还原
cfg_path = sm.SKILL_CONFIG_PATH
cfg_backup = cfg_path.read_text(encoding='utf-8') if cfg_path.exists() else None

try:
    with tempfile.TemporaryDirectory() as tmp:
        demo = SkillsManager(pool_dir=Path(tmp))
        demo.init()  # 内置技能照常注册，自定义池为空

        print('[1] 新增自定义技能 study_buddy ...')
        s = demo.create_skill(
            name='study_buddy', display_name='学习助手',
            description='把知识点整理成结构化笔记', category='analysis',
            skill_md='## 技能说明\n请把用户提供的知识点整理成：概念、例子、易错点。')
        print('    ->', s['name'], '| builtin =', s['builtin'], '| category =', s['category'])

        print('[2] 列表确认 ...')
        print('    ->', [x['name'] for x in demo.list_skills()])

        print('[3] 编辑技能描述 ...')
        s2 = demo.update_skill('study_buddy', description='整理学习笔记并给出记忆技巧')
        print('    ->', s2['description'])

        print('[4] 开关：禁用再启用 ...')
        s3 = demo.set_enabled('study_buddy', False)
        print('    -> enabled =', s3['enabled'])

        print('[5] 禁用后试运行 -> 抛错 ...')
        try:
            await demo.run_skill('study_buddy', '微积分')
            print('    [NO] 禁用技能竟然能运行')
        except ValueError as exc:
            print('    [OK] 已被拒绝:', exc)

        print('[6] 删除自定义技能 ...')
        demo.delete_skill('study_buddy')
        print('    -> 剩余:', [x['name'] for x in demo.list_skills()])

        print('[7] 删除内置技能 -> 抛错 ...')
        try:
            demo.delete_skill('mock_search')
            print('    [NO] 内置技能竟然被删了')
        except ValueError as exc:
            print('    [OK] 已被拒绝:', exc)

finally:
    # 还原真实配置，并重新初始化全局技能池
    if cfg_backup is None:
        cfg_path.unlink(missing_ok=True)
    else:
        cfg_path.write_text(cfg_backup, encoding='utf-8')
    sm.manager.init()
    print()
    print('[还原] 全局技能池已恢复:', [x['name'] for x in sm.manager.list_skills()])

[1] 新增自定义技能 study_buddy ...
    -> study_buddy | builtin = False | category = analysis
[2] 列表确认 ...
    -> ['mock_search', 'mock_analyze', 'mock_write', 'mock_review', 'study_buddy']
[3] 编辑技能描述 ...
    -> 整理学习笔记并给出记忆技巧
[4] 开关：禁用再启用 ...
    -> enabled = False
[5] 禁用后试运行 -> 抛错 ...
    [OK] 已被拒绝: 技能已禁用：study_buddy
[6] 删除自定义技能 ...
    -> 剩余: ['mock_search', 'mock_analyze', 'mock_write', 'mock_review']
[7] 删除内置技能 -> 抛错 ...
    [OK] 已被拒绝: 内置技能不可删除：mock_search

[还原] 全局技能池已恢复: ['mock_search', 'mock_analyze', 'mock_write', 'mock_review', 'humanizer']


In [14]:
# ============================================================
# Cell 13：隔离演示 —— 用内存构造的 zip 导入技能
# ============================================================
import io, zipfile

cfg_backup = cfg_path.read_text(encoding='utf-8') if cfg_path.exists() else None

def make_zip(entries):
    buf = io.BytesIO()
    with zipfile.ZipFile(buf, 'w', zipfile.ZIP_DEFLATED) as zf:
        for name, content in entries:
            zf.writestr(name, content)
    return buf.getvalue()

try:
    with tempfile.TemporaryDirectory() as tmp:
        demo2 = SkillsManager(pool_dir=Path(tmp))
        demo2.init()

        # 一个合法的技能包：顶层目录 + SKILL.md（兼容 Claude 技能包命名）
        md = ("""---\nname: meeting_minutes\ndisplay_name: 会议纪要\n\ndescription: 把会议对话整理成纪要\ncategory: writing\nversion: 1.0.0\n---\n\n## 技能说明\n请整理会议纪要：议题、结论、待办与负责人。\n""")
        data = make_zip([
            ('meeting_minutes/SKILL.md', md),
            ('meeting_minutes/README.md', '会议纪要技能包示例'),
        ])

        print('[1] 导入合法 zip（含 SKILL.md）...')
        s = demo2.import_skill(data)
        print('    ->', s['name'], '| category =', s['category'], '| builtin =', s['builtin'])

        # 路径穿越攻击包：../evil.py 试图逃出解压目录
        print('[2] 导入恶意 zip（../ 路径穿越）-> 拒绝 ...')
        evil_md = md.replace('meeting_minutes', 'evil')
        evil = make_zip([('skill.md', evil_md), ('../evil.py', 'print("hacked")')])
        try:
            demo2.import_skill(evil)
            print('    [NO] 恶意包竟然通过')
        except ValueError as exc:
            print('    [OK] 已被拒绝:', exc)

        # 解压后超过 2MB 的包
        print('[3] 导入超大 zip（解压后 >2MB）-> 拒绝 ...')
        big = make_zip([('skill.md', evil_md), ('padding.bin', '0' * (2 * 1024 * 1024 + 1))])
        try:
            demo2.import_skill(big)
            print('    [NO] 超大包竟然通过')
        except ValueError as exc:
            print('    [OK] 已被拒绝:', exc)

finally:
    if cfg_backup is None:
        cfg_path.unlink(missing_ok=True)
    else:
        cfg_path.write_text(cfg_backup, encoding='utf-8')
    sm.manager.init()
    print()
    print('[还原] 全局技能池已恢复:', [x['name'] for x in sm.manager.list_skills()])

[1] 导入合法 zip（含 SKILL.md）...
    -> meeting_minutes | category = writing | builtin = False
[2] 导入恶意 zip（../ 路径穿越）-> 拒绝 ...
    [OK] 已被拒绝: 技能包包含非法路径，已拒绝导入
[3] 导入超大 zip（解压后 >2MB）-> 拒绝 ...
    [OK] 已被拒绝: 技能包解压后体积过大

[还原] 全局技能池已恢复: ['mock_search', 'mock_analyze', 'mock_write', 'mock_review', 'humanizer']


---
## Part 8：接入 API 与前端（技能中心）
---

### 8.1 后端 API

`backend/api/routes.py` 注册了 `/api/v1/skills` 系列接口（见 Cell 14）：
列表 / 新增 / 编辑 / 删除 / 开关 / 导入 / 试运行。

请求模型层有两处校验值得注意（`routes.py`）：

- `SkillCreateRequest.name` 用正则 `^[a-zA-Z0-9_-]+$`（API 层宽松校验）
- `category` 用 `^(search|analysis|writing|review|other)$`（非法值直接 422）

### 8.2 前端「技能中心」（本期只作了解）

`frontend/src/views/SkillCenter.vue` 提供技能卡片网格 + 新增 / 编辑 / 删除 / 导入对话框，
交互逻辑与第 22 节「工具监控」页是同一套（Element Plus 表格 + 对话框 + `el-upload`）。

> 前端不是本节课重点：能看懂「卡片 = 后端 `list_skills()` 返回数据的渲染」即可。

In [15]:
# ============================================================
# Cell 14：把技能接进 API —— 路由一览 + 当前列表
# ============================================================
from api.routes import router

print('=' * 55)
print('  /skills 相关路由')
print('=' * 55)
for route in router.routes:
    if hasattr(route, 'path') and 'skill' in route.path:
        methods = ','.join(sorted(route.methods - {'HEAD', 'OPTIONS'}))
        print(f'  {methods:12s} {route.path}')

print()
print('=' * 55)
print('  当前技能池列表（内置 + 自定义）')
print('=' * 55)
for s in sm.manager.list_skills():
    print(f'  {s["name"]:14s} [{s["category"]:8s}] builtin={s["builtin"]} enabled={s["enabled"]}')

print()
print('[提示] 想体验完整 API？启动后端后访问：')
print('       GET  http://localhost:8002/api/v1/skills')

  /skills 相关路由
  GET          /api/v1/skills
  POST         /api/v1/skills
  PUT          /api/v1/skills/{name}
  DELETE       /api/v1/skills/{name}
  PUT          /api/v1/skills/{name}/enabled
  POST         /api/v1/skills/import
  POST         /api/v1/skills/{name}/run

  当前技能池列表（内置 + 自定义）
  mock_search    [search  ] builtin=True enabled=True
  mock_analyze   [analysis] builtin=True enabled=True
  mock_write     [writing ] builtin=True enabled=True
  mock_review    [review  ] builtin=True enabled=True
  humanizer      [analysis] builtin=False enabled=True

[提示] 想体验完整 API？启动后端后访问：
       GET  http://localhost:8002/api/v1/skills


---
## Part 9：课程总结
---

### 这节课我们经历了什么？

| 演进阶段 | 遇到的问题 | 解决方案 |
|---------|-----------|---------|
| v0 | 技能散落，入参出参各写各的 | 统一抽象：SkillInput / SkillOutput / BaseSkill |
| v1 | 技能无法统一管理 | 注册表 SkillRegistry（重名拦截） |
| v2 | 内置 / 自定义傻傻分不清 | builtin 标记：内置只读，自定义可管 |
| v3 | 每个技能都要写代码 | skill.md 配置化：**文档即技能**（MarkdownSkill） |
| v4 | 技能会越来越多，管不过来 | SkillsManager：CRUD / zip 导入 / 开关持久化 |

### 关键设计原则

1. **契约优先**：统一的输入 / 输出 / 元信息，上层才能一致处理
2. **注册表模式**：集中注册、统一路由（与第 22 节工具注册表一脉相承）
3. **系统 vs 用户边界**：内置技能只读，自定义技能自由管理
4. **文档即能力**：skill.md 是技能的定义，代码只是可选实现
5. **配置与能力分离**：开关状态单独存 `skill_config.json`，不污染技能文件
6. **纵深防御**：zip 导入双重大小校验 + 路径穿越逐条拦截

### 下节课预告

**第 24 节：多智能体架构与 Plan-and-Execute 规划落地**

- 引入 LangGraph：Planner 生成计划 → Scheduler 派发 → ReAct Researcher/Writer 执行
- 第 23 节的技能池将成为 Agent 的「能力弹药库」

---
## 课后练习
---

### 练习一：新增一个自定义技能

参考 Cell 12，用 `SkillsManager` 注册一个「邮件助手」技能：

1. `name = "email_helper"`，分类 `writing`
2. skill.md 正文写清楚：把零散要点整理成主题 / 称呼 / 正文 / 署名
3. 在隔离临时目录里完成注册与列表确认（**不要污染真实技能池**）

### 练习二：手工构造 zip 技能包

用 `io.BytesIO` + `zipfile` 构造技能包并走一遍导入校验（参考 Cell 13）：

1. 构造一个合法包（根目录或唯一顶层目录 + skill.md），确认导入成功
2. 构造一个含 `../../evil.py` 的恶意包，确认被拒绝
3. 思考：为什么要兼容 zip 里的 `SKILL.md` 大写命名？（提示：Claude 技能包）

### 练习三（思考题）：技能名能支持中文吗？

项目默认 `NAME_RE` 只允许英文。想想如果放开中文名：

1. URL 路径 `/skills/{name}` 需要做什么处理？
2. 文件系统 `skills_pool/<name>/` 有哪些潜在坑？
3. 你会怎么权衡「易用性」与「安全」？

In [16]:
# ============================================================
# Cell 15：练习参考 —— 练习一：注册「邮件助手」技能
# ============================================================
import tempfile

cfg_backup = cfg_path.read_text(encoding='utf-8') if cfg_path.exists() else None

try:
    with tempfile.TemporaryDirectory() as tmp:
        demo3 = SkillsManager(pool_dir=Path(tmp))
        demo3.init()

        # 1. 创建技能：name / display_name / description / category / skill_md
        s = demo3.create_skill(
            name='email_helper', display_name='邮件助手',
            description='把零散要点整理成专业邮件', category='writing',
            skill_md='## 技能说明\n把用户给的要点整理成：主题、称呼、正文、署名。')
        print('创建成功:', s['name'], '| builtin =', s['builtin'])

        # 2. 列表确认
        found = [x for x in demo3.list_skills() if x['name'] == 'email_helper'][0]
        print('列表确认:', found['display_name'], '|', found['category'], '|', found['description'])

        # 3. 校验：非法分类被拒绝
        try:
            demo3.create_skill(name='bad_skill', display_name='坏技能', category='hacking')
            print('[NO] 非法分类竟然通过')
        except ValueError as exc:
            print('[OK] 非法分类被拦截:', exc)

finally:
    if cfg_backup is None:
        cfg_path.unlink(missing_ok=True)
    else:
        cfg_path.write_text(cfg_backup, encoding='utf-8')
    sm.manager.init()

print()
print('=' * 55)
print('  练习三参考：技能名支持中文？先看正则差异')
print('=' * 55)
import re
chinese_name_re = re.compile(r'^[\u4e00-\u9fa5A-Za-z0-9][\u4e00-\u9fa5A-Za-z0-9_-]{0,63}$')
for n in ['邮件助手', 'email_helper', 'hello-world', 'a b', '中文_1']:
    match = 'OK  ' if chinese_name_re.match(n) else '拒绝'
    print(f'  {n:14s} -> {match}')

print()
print('[提示] 项目默认 NAME_RE 只允许英文。放开中文名要权衡：')
print('       1. URL 路径 /skills/{name} 需要 URL 编码与解码')
print('       2. 文件系统目录名 + Windows 大小写不敏感都是坑')
print('       3. 教学项目选择「保守优先」：名字宁严勿松')

创建成功: email_helper | builtin = False
列表确认: 邮件助手 | writing | 把零散要点整理成专业邮件
[OK] 非法分类被拦截: 分类必须是 search、analysis、writing、review、other 之一

  练习三参考：技能名支持中文？先看正则差异
  邮件助手           -> OK  
  email_helper   -> OK  
  hello-world    -> OK  
  a b            -> 拒绝
  中文_1           -> OK  

[提示] 项目默认 NAME_RE 只允许英文。放开中文名要权衡：
       1. URL 路径 /skills/{name} 需要 URL 编码与解码
       2. 文件系统目录名 + Windows 大小写不敏感都是坑
       3. 教学项目选择「保守优先」：名字宁严勿松


---

## 📚 参考资料

- 本节需求文档：《多智能体协作开发PRD（拆分）.md》第 23 节（项目根目录）
- 技能抽象：`backend/skills/base_skill.py`
- 技能注册表：`backend/skills/registry.py`
- 内置技能：`backend/skills/mock_skills.py`
- 技能管理器：`backend/skills/skills_manager.py`
- 技能 API：`backend/api/routes.py`
- 示例自定义技能：`backend/skills_pool/humanizer/skill.md`
- 前端技能中心：`frontend/src/views/SkillCenter.vue`

> 🎉 恭喜你完成了本节课的学习！
> 现在你不仅知道技能池的代码长什么样，更知道它为什么这么设计。